# Taxis NYC: pronóstico, segmentación y decisión

Este notebook prepara los artefactos usados por la API y la app. La unidad de pronóstico es zona-hora; el entrenamiento termina el 24 de enero y la prueba cubre del 25 al 31.

In [1]:
from pathlib import Path
import pandas as pd
from preparacion import load_trips, make_series, train_forecast, segment_zones

## 1. Datos y filtros comunes

In [2]:
trips, zones = load_trips()
print('Viajes válidos:', len(trips))
print('Zonas disponibles:', trips['PULocationID'].nunique())
trips.head()

Viajes válidos: 2954246
Zonas disponibles: 259


,tpep_pickup_datetime,PULocationID,trip_distance,fare_amount,tip_amount,hora
0,2024-01-01 00:57:55,186,1.72,17.7,0.00,2024-01-01
1,2024-01-01 00:03:00,140,1.80,10.0,3.75,2024-01-01
2,2024-01-01 00:17:06,236,4.70,23.3,3.00,2024-01-01
3,2024-01-01 00:36:38,79,1.40,10.0,2.00,2024-01-01
4,2024-01-01 00:46:51,211,0.80,7.9,3.20,2024-01-01


## 2. Serie horaria y variables pasadas

In [3]:
series, zonas_pronosticables = make_series(trips)
print('Zonas pronosticables:', len(zonas_pronosticables))
print('Filas de la serie:', len(series))
series.head()

Zonas pronosticables: 58
Filas de la serie: 44544


,PULocationID,hora,viajes,lag_24,lag_48,lag_167,lag_168,lag_169,media_dia_ant,hora_num,hora_del_dia,dia_semana,feriado
0,4,2024-01-01 00:00:00,25,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0
1,4,2024-01-01 01:00:00,29,NaN,NaN,NaN,NaN,NaN,NaN,1,1,0,0
2,4,2024-01-01 02:00:00,34,NaN,NaN,NaN,NaN,NaN,NaN,2,2,0,0
3,4,2024-01-01 03:00:00,31,NaN,NaN,NaN,NaN,NaN,NaN,3,3,0,0
4,4,2024-01-01 04:00:00,32,NaN,NaN,NaN,NaN,NaN,NaN,4,4,0,0


## 3. Modelo de cuantiles y evaluación temporal

In [4]:
modelo, demanda, metricas = train_forecast(series)
print(pd.Series(metricas))
print('El baseline de hace una semana se conserva como comparación; si su MAE es menor, el modelo debe comunicarlo abiertamente.')

mae_p50                   12.144484
mae_hace_una_semana       11.881773
cobertura_p10_p90          0.874384
filas_entrenamiento    23606.000000
filas_prueba            9744.000000
dtype: float64
El baseline de hace una semana se conserva como comparación; si su MAE es menor, el modelo debe comunicarlo abiertamente.


## 4. Segmentación de zonas con K-Means y PCA

In [5]:
zonas_segmentadas, siluetas, varianza_pca = segment_zones(trips, zones)
print('Mejor k:', max(siluetas, key=lambda x: x['silueta']))
print('Varianza conservada por PCA:', sum(varianza_pca))
zonas_segmentadas[['PULocationID', 'nombre', 'distrito', 'grupo_nombre', 'pca_1', 'pca_2']].head()

Mejor k: {'k': 5, 'silueta': 0.4217428351909625}
Varianza conservada por PCA: 0.6558730677230404


,PULocationID,nombre,distrito,grupo_nombre,pca_1,pca_2
0,4,Alphabet City,Manhattan,Vida nocturna,-0.195287,4.535425
1,7,Astoria,Queens,Demanda mixta,-1.041184,0.337627
2,10,Baisley Park,Queens,Demanda mixta,-3.378980,-0.008703
3,12,Battery Park,Manhattan,Demanda mixta,0.208413,-0.260423
4,13,Battery Park City,Manhattan,Demanda mixta,1.701693,-1.586851


## 5. Decisión por costos

El cuantil óptimo se calcula como `costo_perdido / (costo_perdido + costo_ocioso)`. Con costos de 40 por demanda perdida y 10 por vehículo ocioso, el cálculo lleva a trabajar con el cuantil p80.

In [6]:
costo_perdido, costo_ocioso = 40, 10
cuantil_optimo = costo_perdido / (costo_perdido + costo_ocioso)
print('Cuantil óptimo:', cuantil_optimo)

Cuantil óptimo: 0.8


## 6. Artefactos generados

El modelo, la demanda con las variables calculadas, las zonas segmentadas y las métricas quedan guardados en `api/` y `app/data/`. Así la API y la aplicación pueden responder sin repetir todo el procesamiento cada vez que se consultan.

## Cierre

El modelo de cuantiles obtuvo un MAE p50 de 12.14 y una cobertura de 87.4% para el intervalo p10–p90. Como referencia, el baseline de la semana anterior obtuvo un MAE ligeramente menor, de 11.88; por eso el modelo todavía debe mejorarse antes de presentarlo como una mejora definitiva. Aun así, la segmentación y la API quedaron preparadas para explorar la demanda por zona y apoyar decisiones de operación.